In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler

presion 996.52
tempeartura -8.02

X = [(996.52,-8.02),(996.57,8.41),(996.53,-8.51)]
y = -8.31

In [2]:
df = pd.read_csv('/content/jena_climate_2009_2016.csv')
df

,Date Time,p (mbar),T (degC),Tpot (K),Tdew (degC),rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),H2OC (mmol/mol),rho (g/m**3),wv (m/s),max. wv (m/s),wd (deg)
0,01.01.2009 00:10:00,996.52,-8.02,265.40,-8.90,93.30,3.33,3.11,0.22,1.94,3.12,1307.75,1.03,1.75,152.3
1,01.01.2009 00:20:00,996.57,-8.41,265.01,-9.28,93.40,3.23,3.02,0.21,1.89,3.03,1309.80,0.72,1.50,136.1
2,01.01.2009 00:30:00,996.53,-8.51,264.91,-9.31,93.90,3.21,3.01,0.20,1.88,3.02,1310.24,0.19,0.63,171.6
3,01.01.2009 00:40:00,996.51,-8.31,265.12,-9.07,94.20,3.26,3.07,0.19,1.92,3.08,1309.19,0.34,0.50,198.0
4,01.01.2009 00:50:00,996.51,-8.27,265.15,-9.04,94.10,3.27,3.08,0.19,1.92,3.09,1309.00,0.32,0.63,214.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
420546,31.12.2016 23:20:00,1000.07,-4.05,269.10,-8.13,73.10,4.52,3.30,1.22,2.06,3.30,1292.98,0.67,1.52,240.0
420547,31.12.2016 23:30:00,999.93,-3.35,269.81,-8.06,69.71,4.77,3.32,1.44,2.07,3.32,1289.44,1.14,1.92,234.3
420548,31.12.2016 23:40:00,999.82,-3.16,270.01,-8.21,67.91,4.84,3.28,1.55,2.05,3.28,1288.39,1.08,2.00,215.2
420549,31.12.2016 23:50:00,999.81,-4.23,268.94,-8.53,71.80,4.46,3.20,1.26,1.99,3.20,1293.56,1.49,2.16,225.8


In [3]:
df.columns

Index(['Date Time', 'p (mbar)', 'T (degC)', 'Tpot (K)', 'Tdew (degC)',
       'rh (%)', 'VPmax (mbar)', 'VPact (mbar)', 'VPdef (mbar)', 'sh (g/kg)',
       'H2OC (mmol/mol)', 'rho (g/m**3)', 'wv (m/s)', 'max. wv (m/s)',
       'wd (deg)'],
      dtype='object')

In [4]:
# Seleccionar dos variables para X y una para Y
features = ["p (mbar)", "T (degC)"]  # Dos variables de entrada (Presión y Temperatura)
target = "T (degC)"  # Variable de salida (Temperatura)

In [5]:
# Normalizar los datos
scaler_X = MinMaxScaler()
scaler_Y = MinMaxScaler()
df[features] = scaler_X.fit_transform(df[features])
df[target] = scaler_Y.fit_transform(df[[target]])

In [7]:
# Convertir en secuencias para series temporales
def create_sequences(data, target, seq_length):
    X, Y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])  # Secuencia de longitud `seq_length`
        Y.append(target[i+seq_length])  # Valor siguiente de la serie
    return np.array(X), np.array(Y)

In [12]:
seq_length = 30  # Número de pasos en el tiempo
X_data, Y_data = create_sequences(df[features].values, df[target].values, seq_length)

In [11]:
print(X_data[0],Y_data[0])

[[0.81493857 0.24863161]
 [0.81542998 0.24216288]
 [0.81503686 0.24050423]] 0.24382152927517003


In [13]:
# Dividir en conjunto de entrenamiento y prueba
split = int(len(X_data) * 0.8)
X_train, X_test = X_data[:split], X_data[split:]
Y_train, Y_test = Y_data[:split], Y_data[split:]

In [14]:
# Crear el modelo LSTM
model = Sequential([
    LSTM(50, activation='relu', return_sequences=True, input_shape=(seq_length, len(features))),
    LSTM(50, activation='relu'),
    Dense(1)  # Una sola salida
])
model.compile(optimizer='adam', loss='mse')

/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
# Entrenar el modelo
model.fit(X_train, Y_train, epochs=10, batch_size=30, validation_data=(X_test, Y_test))


Epoch 1/10
10282/11214 ━━━━━━━━━━━━━━━━━━━━ 28s 31ms/step - loss: 0.0028

In [ ]:
y_pred = model.predict(X_test)
scaler_y = MinMaxScaler()
scaler_y.fit(df[:, [2]])
y_test_inv = scaler_y.inverse_transform(Y_test.reshape(-1, 1))
y_pred_inv = scaler_y.inverse_transform(y_pred)

In [ ]:
import matplotlib.pyplot as plt
# Graficar predicciones vs valores reales
plt.figure(figsize=(10, 5))
plt.plot(y_test_inv, label='Real', linewidth=2)
plt.plot(y_pred_inv, label='Predicción', linestyle='dashed', linewidth=2)
plt.legend()
plt.title("Predicción de serie temporal bivariada con LSTM")
plt.show()